In [19]:
import pandas as pd
import numpy as np
import warnings
import time
import joblib
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score,
    classification_report, confusion_matrix
)

In [20]:
train_df = pd.read_csv("cleaned_train.xls")
test_df  = pd.read_csv("cleaned_test.xls")

X_train = train_df['final_text']
y_train = train_df['category']
X_test  = test_df['final_text']
y_test  = test_df['category']


In [21]:
train_df.drop_duplicates(inplace=True)
test_df.drop_duplicates(inplace=True)
print(f"   Train duplicates  : {train_df.duplicated().sum()}")
print(f"   Test  duplicates  : {test_df.duplicated().sum()}")

   Train duplicates  : 0
   Test  duplicates  : 0


In [22]:
tfidf = TfidfVectorizer(
    max_features=30000,     
    ngram_range=(1, 2),    
    sublinear_tf=True,      
    min_df=2,              
    max_df=0.95             
)

X_train_vec = tfidf.fit_transform(X_train)
X_test_vec  = tfidf.transform(X_test)

In [23]:
print(f"   Vocabulary size : {len(tfidf.vocabulary_):,} features")
print(f"   Train matrix    : {X_train_vec.shape}")
print(f"   Test  matrix    : {X_test_vec.shape}")


   Vocabulary size : 30,000 features
   Train matrix    : (103800, 30000)
   Test  matrix    : (25964, 30000)


In [24]:
all_results = {}
def train_evaluate(name, model, X_tr, y_tr, X_te, y_te, grid_params=None):
    """
    Train a model (optionally with GridSearch), evaluate on test set,
    and return metrics.
    """
    print(f"\n{'─'*65}")
    print(f"  MODEL: {name}")
    print(f"{'─'*65}")

    # ── Grid Search ──
    if grid_params:
        print(f"   GridSearchCV | params: {grid_params}")
        gs = GridSearchCV(
            model,
            grid_params,
            cv=3,
            scoring='f1_weighted',
            n_jobs=2,
            verbose=0
        )
        gs.fit(X_tr, y_tr)
        best_params = gs.best_params_
        cv_score    = gs.best_score_
        print(f"   Best Params  : {best_params}")
        print(f"   CV F1-Score  : {cv_score:.4f}")

        # Re-create model with best params for final training
        model.set_params(**best_params)

    # ── Train final model ──
    t0 = time.time()
    model.fit(X_tr, y_tr)
    train_time = time.time() - t0

    # ── Predict & Evaluate ──
    y_pred = model.predict(X_te)

    acc  = accuracy_score(y_te, y_pred)
    prec = precision_score(y_te, y_pred, average='weighted')
    rec  = recall_score(y_te, y_pred, average='weighted')
    f1   = f1_score(y_te, y_pred, average='weighted')

    print(f"\n   Test Set Results:")
    print(f"     Accuracy  : {acc:.4f}  ({acc*100:.2f}%)")
    print(f"     Precision : {prec:.4f}")
    print(f"     Recall    : {rec:.4f}")
    print(f"     F1-Score  : {f1:.4f}")
    print(f"     Train Time: {train_time:.2f}s")

    print(f"\n   Per-Class Report:")
    print(classification_report(y_te, y_pred,
                                target_names=sorted(y_te.unique())))

    # Save results
    all_results[name] = {
        'Accuracy':       round(acc,  4),
        'Precision':      round(prec, 4),
        'Recall':         round(rec,  4),
        'F1-Score':       round(f1,   4),
        'Best Params':    best_params if grid_params else 'default',
        'Train Time (s)': round(train_time, 2),
        'model':          model,
        'y_pred':         y_pred
    }

    return model



In [25]:

nb_model = train_evaluate(
    name        = "Naive Bayes",
    model       = MultinomialNB(),
    X_tr        = X_train_vec,
    y_tr        = y_train,
    X_te        = X_test_vec,
    y_te        = y_test,
    grid_params = {'alpha': [0.01, 0.1, 0.5, 1.0, 2.0]}
)


─────────────────────────────────────────────────────────────────
  MODEL: Naive Bayes
─────────────────────────────────────────────────────────────────
   GridSearchCV | params: {'alpha': [0.01, 0.1, 0.5, 1.0, 2.0]}
   Best Params  : {'alpha': 0.1}
   CV F1-Score  : 0.9069

   Test Set Results:
     Accuracy  : 0.9098  (90.98%)
     Precision : 0.9096
     Recall    : 0.9098
     F1-Score  : 0.9096
     Train Time: 0.56s

   Per-Class Report:
              precision    recall  f1-score   support

    Business       0.88      0.87      0.88      6369
      Sports       0.95      0.98      0.97      6458
        Tech       0.88      0.89      0.89      6526
       World       0.92      0.90      0.91      6611

    accuracy                           0.91     25964
   macro avg       0.91      0.91      0.91     25964
weighted avg       0.91      0.91      0.91     25964



In [26]:

lr_model = train_evaluate(
    name        = "Logistic Regression",
    model       = LogisticRegression(max_iter=500),
    X_tr        = X_train_vec,
    y_tr        = y_train,
    X_te        = X_test_vec,
    y_te        = y_test,
    grid_params = {
        'C':       [0.1, 1.0, 5.0, 10.0],
        'solver':  ['lbfgs', 'saga']
    }
)


─────────────────────────────────────────────────────────────────
  MODEL: Logistic Regression
─────────────────────────────────────────────────────────────────
   GridSearchCV | params: {'C': [0.1, 1.0, 5.0, 10.0], 'solver': ['lbfgs', 'saga']}
   Best Params  : {'C': 1.0, 'solver': 'lbfgs'}
   CV F1-Score  : 0.9153

   Test Set Results:
     Accuracy  : 0.9199  (91.99%)
     Precision : 0.9199
     Recall    : 0.9199
     F1-Score  : 0.9198
     Train Time: 20.39s

   Per-Class Report:
              precision    recall  f1-score   support

    Business       0.88      0.90      0.89      6369
      Sports       0.96      0.98      0.97      6458
        Tech       0.90      0.90      0.90      6526
       World       0.94      0.90      0.92      6611

    accuracy                           0.92     25964
   macro avg       0.92      0.92      0.92     25964
weighted avg       0.92      0.92      0.92     25964



In [27]:

rf_model = train_evaluate(
    name        = "Random Forest",
    model       = RandomForestClassifier(random_state=42, n_jobs=2),
    X_tr        = X_train_vec,
    y_tr        = y_train,
    X_te        = X_test_vec,
    y_te        = y_test,
    grid_params = {
        'n_estimators': [50, 100],
        'max_depth':    [20, 30]
    }
)


─────────────────────────────────────────────────────────────────
  MODEL: Random Forest
─────────────────────────────────────────────────────────────────
   GridSearchCV | params: {'n_estimators': [50, 100], 'max_depth': [20, 30]}
   Best Params  : {'max_depth': 30, 'n_estimators': 100}
   CV F1-Score  : 0.8430

   Test Set Results:
     Accuracy  : 0.8508  (85.08%)
     Precision : 0.8514
     Recall    : 0.8508
     F1-Score  : 0.8496
     Train Time: 22.67s

   Per-Class Report:
              precision    recall  f1-score   support

    Business       0.84      0.80      0.82      6369
      Sports       0.84      0.97      0.90      6458
        Tech       0.84      0.79      0.82      6526
       World       0.88      0.84      0.86      6611

    accuracy                           0.85     25964
   macro avg       0.85      0.85      0.85     25964
weighted avg       0.85      0.85      0.85     25964



In [28]:

comparison = {
    k: {m: v for m, v in r.items() if m not in ['model', 'y_pred', 'Best Params']}
    for k, r in all_results.items()
}
df_comp = pd.DataFrame(comparison).T.drop(columns=['Train Time (s)'], errors='ignore')
print(f"\n{df_comp.to_string()}")


                     Accuracy  Precision  Recall  F1-Score
Naive Bayes            0.9098     0.9096  0.9098    0.9096
Logistic Regression    0.9199     0.9199  0.9199    0.9198
Random Forest          0.8508     0.8514  0.8508    0.8496


In [29]:
best_name = df_comp['F1-Score'].astype(float).idxmax()
best_f1   = all_results[best_name]['F1-Score']
best_acc  = all_results[best_name]['Accuracy']
print(f"     BEST MODEL  : {best_name}")
print(f"     F1-Score   : {best_f1}")
print(f"     Accuracy   : {best_acc}")
print(f"     Best Params: {all_results[best_name]['Best Params']}")



     BEST MODEL  : Logistic Regression
     F1-Score   : 0.9198
     Accuracy   : 0.9199
     Best Params: {'C': 1.0, 'solver': 'lbfgs'}


In [30]:
best_model_obj = all_results[best_name]['model']

joblib.dump(best_model_obj, "best_model.pkl")
joblib.dump(tfidf,          "tfidf_vectorizer.pkl")


['tfidf_vectorizer.pkl']

In [31]:
def predict_news_category(text: str, model=None, vectorizer=None):
    """
    Predict the category of a news article.

    Args:
        text       : The news article text (cleaned/preprocessed)
        model      : Trained classifier (default: best_model_obj)
        vectorizer : Fitted TF-IDF vectorizer (default: tfidf)

    Returns:
        dict with 'category' and 'confidence'
    """
    if model is None:
        model = best_model_obj
    if vectorizer is None:
        vectorizer = tfidf

    vec   = vectorizer.transform([text])
    pred  = model.predict(vec)[0]
    proba = model.predict_proba(vec)[0] if hasattr(model, 'predict_proba') else None
    conf  = max(proba) if proba is not None else None

    return {'category': pred, 'confidence': round(conf, 4) if conf else None}

In [35]:

sample_texts = [
    "The government introduced new tax policies to help small companies grow and attract more investors",
    "Real Madrid secured a dramatic victory in the final minutes of the match to win the title",
    "Researchers developed a new AI system that can detect diseases earlier than doctors",
    "Oil prices surged after global supply disruptions raised concerns among investors",
    "A startup launched a mobile app that helps users manage their daily tasks using machine learning"
]
for text in sample_texts:
    result = predict_news_category(text)
    conf_str = f"  (confidence: {result['confidence']:.1%})" if result['confidence'] else ""
    print(f"  [{result['category']:10s}]{conf_str}")
    print(f"   → {text[:75]}...")
    print()

  [Business  ]  (confidence: 54.0%)
   → The government introduced new tax policies to help small companies grow and...

  [Sports    ]  (confidence: 99.2%)
   → Real Madrid secured a dramatic victory in the final minutes of the match to...

  [Tech      ]  (confidence: 87.8%)
   → Researchers developed a new AI system that can detect diseases earlier than...

  [Business  ]  (confidence: 84.6%)
   → Oil prices surged after global supply disruptions raised concerns among inv...

  [Tech      ]  (confidence: 89.3%)
   → A startup launched a mobile app that helps users manage their daily tasks u...

